# 02 本地校验（上传前必做）

逐分片解析 safetensors 表头（结构损坏会抛错）、检查关键配置文件、记录体积。

In [1]:
from safetensors import safe_open
import glob, os, json

LOCAL = os.path.expanduser("~/models/parallax")
EXPECT = {  # 名称 -> 预期最小体积 GB
    "Qwen3-4B-Instruct-2507": 8.0,
    "Qwen3-32B-AWQ": 18.0,
    "Qwen3-8B": 15.0,
}
ok = True
for name, min_gb in EXPECT.items():
    d = os.path.join(LOCAL, name)
    sts = sorted(glob.glob(f"{d}/*.safetensors"))
    assert sts, f"{name} 没有 safetensors，下载不完整"
    for s in sts:
        with safe_open(s, framework="pt") as f:
            _ = list(f.keys())
    gb = sum(os.path.getsize(x)/1e9 for x in sts)
    has_cfg = os.path.exists(f"{d}/config.json")
    good = gb >= min_gb and has_cfg
    ok &= good
    print(f"{name:20s} 分片{len(sts):3d}  体积{gb:6.2f}GB  config:{has_cfg}  -> {'PASS' if good else 'FAIL'}")
assert ok, "存在不完整模型，先补下载"
print("全部通过")

Qwen3-4B-Instruct-2507 分片  3  体积  8.04GB  config:True  -> PASS
Qwen3-32B-AWQ        分片  4  体积 19.33GB  config:True  -> PASS
Qwen3-8B             分片  5  体积 16.38GB  config:True  -> PASS
全部通过


In [2]:
# GGUF 体积核对（P2 可选）
import glob, os
for f in glob.glob(os.path.expanduser("~/models/parallax") + "/*.gguf"):
    print(os.path.basename(f), round(os.path.getsize(f)/1e9, 2), "GB")

In [3]:
# Apple Silicon 冒烟（可选，MLX 加载 4B 验证权重可读；内存 <16GB 的机器跳过）
try:
    from mlx_lm import load, generate
    m, t = load(os.path.expanduser("~/models/parallax/Qwen3-4B-Instruct-2507"))
    print(generate(m, t, prompt="用一句话介绍你自己", max_tokens=16))
except Exception as e:
    print("跳过 MLX 冒烟:", e)

跳过 MLX 冒烟: No module named 'mlx_lm'
